In [ ]:
### schemas.py

from pydantic import BaseModel, Field
from typing import List, Dict

class ADMETSummary(BaseModel):
    """Agent 1 Output: Distilled canonical ADMET facts."""

    drug_name: str = Field(
        description='The standard name of the evaluated drug.'
    )
    model_avg_score: float = Field(
        description="Prior computational model predicted interaction probability (5.0 to 10.0)."
    )

    # 0. Categories list of keywords
    category_tags: List[str] = Field(
        description="Complete list of cleaned, high-level ontology/category tags from input data."
    )
    # 1. Combined Background & Indications (Description + Indication)
    clinical_overview: str = Field(
        description=("1-2 sentence summary of the drug's therapeutic class, core description) and approved disease indications. Output 'UNKNOWN' if missing or unspecified.")
    )

    # 2. Pharmacodynamics & Target Binding (Targets + Pharmacodynamics)
    pharmacodynamics_and_targets: str = Field(
        description=("Summary of primary therapeutic targets (enzymes, receptors and affected-organisms - if not 'Human and other mammals'), mechanism of action, target binding selectivity, pharmacodynamics, and biological response. Output 'UNKNOWN' if missing or unspecified.")
    )

    # 3. Pharmacokinetics / ADME (PK fields + Protein Binding + Route of Elimination)
    pk_properties: str = Field(
        description=("Summary of ADME: absorption, plasma protein binding (PPB), volume of distribution (Vd), metabolism (CYP enzymes), half-life, clearance, and route of elimination. Output 'UNKNOWN' if missing or unspecified.")
    )

    # 4. Toxicity & Safety Highlights (Toxicity)
    toxicity_highlights: List[str] = Field(
        description=("Bullet points of key toxicities, major organ toxicities (e.g., cardiotoxicity, hepatotoxicity), black box warnings, narrow therapeutic, or adverse category flags. Output ['UNKNOWN'] if missing or unspecified.")
    )

    # 5. Distilled Dosage Feasibility (Dosages)
    dosage_and_route_summary: List[str] = Field(
        description=("Concise bullet points listing available administration routes (oral, IV, etc.), dosing forms, and any notable dosing constraints. Avoid repetition and redundancies. Output ['UNKNOWN'] if missing or unspecified.")
        # description=("Concise bullet points listing ONLY available administration routes (oral, IV, etc.), WITHOUT dosing quantities. Avoid repetition and redundancies.")
    )

class ADMETEvaluation(BaseModel):
    # drug_name: str
    # --- DQs ---
    vaccine: bool = Field(
        default=False,
        description="True if the drug is actually a vaccine."
    )

    antibody: bool = Field(
        default=False,
        description="True if drug is an antibody (often ending in 'mab' or therapeutic mAb)."
    )

    chemotherapy: bool = Field(
        default=False,
        description=('True if the drug is a chemotherapy agent used in cancer treatment.')
    )
    # True if the drug is a traditional cytotoxic chemotherapy agent used in cancer treatment.

    cytotoxic: bool = Field(
        default=False,
        description="True if the drug is a cytotoxic agent."
    )

    # --- Unfavorables

    immunosuppressor: bool = Field(
        default=False,
        description="True if the drug is an immunosuppressant."
    )

    antiinflammatory: bool = Field(
        default=False,
        description="True if the drug is anti-inflammatory."
    )

    IV: bool = Field(
        default=False,
        description="True if the drug IS ONLY available for intravenous administration."
    )

    # --- Favorables

    oral_inhalation: bool = Field(
        default=False,
        description=('True if the drug is available for oral or inhalation administration.')
    )

    antiviral: bool = Field(
        default=False,
        description=('True if the drug is indicated or used for treating viral infections or targeting viral mechanisms.')
    )

    DNA_virus: bool = Field(
        default=False,
        description="True if the drug is used to treat DNA viruses."
    )

    # --- Desirables

    RNA_virus: bool = Field(
        default=False,
        description="True if the drug is used to treat RNA viruses."
    )

    endo: bool = Field(
        default=False,
        description="True if the drug is used to treat endothelial cell dysfunction, or boost endothelial cell function, or block endothelial inflammation."
    )

    respiratory: bool = Field(
        default=False,
        description="True if the drug is used to treat respiratory dysfunction or boost respiratory function."
    )

    neuronal: bool = Field(
        default=False,
        description="True if the drug is used to treat neuronal dysfunction or encephalitis, or boost neuronal function or are neuroprotective."
    )

    # --- CHAIN OF THOUGHT: Rationales defined BEFORE scores ---
    pharmacodynamics_rationale: str = Field(
        default="",
        description=(
            "Step-by-step reasoning evaluating mechanism of action, target affinity, receptor selectivity, and biochemical efficacy."
        )
    )
    pharmacodynamics_score: float = Field(
        default=0.0,
        ge=0.0,
        le=10.0,
        description=(
            "0.0 (weak target binding/unclear MoA) to 10.0 (high potency, selective target binding, proven efficacy) based on rationale above."
        )
    )
    safety_rationale: str = Field(
        default="",
        description='Step-by-step reasoning for the safety score. Penalize heavily for severe adverse reactions, discontinuation warnings, black box warnings, and severe organ toxicity.'
    )
    safety_score: float = Field(
        default=0.0,
        ge=0.0,
        le=10.0,
        description="0.0 (severe off-target toxicity/cardiotoxicity) to 10.0 (clean safety profile)."
    )

    pharmacokinetics_rationale: str = Field(
        default="",
        description='Step-by-step reasoning for the pharmacokinetics score. Based on half-life, clearance rates, metabolism, volume of distribution, and bioavailability.'
    )
    pharmacokinetics_score: float = Field(
        default=0.0,
        ge=0.0,
        le=10.0,
        description="0.0 (poor bioavailability/unfavorable clearance) to 10.0 (optimal drug-like ADME)."
    )

    admin_routes: List[str] = Field(
        default_factory=list,
        description="List of routes of administration (e.g., Oral, IV, Topical)."
    )

    targeted_viruses: List[str] = Field(
        default_factory=list,
        description="List of viruses target by drug, if drug is antiviral."
    )

    major_red_flags: List[str] = Field(
        default_factory=list,
        description="List of critical clinical concerns, toxicity warnings, or severe drug interaction hazards."
    )

    key_advantages: List[str] = Field(
        default_factory=list,
        description="List of favorable ADMET traits, long half-life, high tolerability, or available for simple oral dosing or inhalation."
    )

    missing_entries: List[str] = Field(
        default_factory=list,
        description="List of missing entries from PD, PK or Safety."
    )

    model_avg_score: float = Field(
        default=0.0,
        description="Prior computational model predicted interaction probability (5.0 to 10.0)."
    )

    overall_prioritization_rationale: str = Field(
        default="",
        description="Synthesized 2-3 sentence clinical rationale explaining why to prioritize or deprioritize this candidate for repurposing. " \
        "* Explicitly evaluate ModelAvg score, Safety, PD, and PK. " \
        # "* If 'model_avg_score'is weak (< 6.0), overall score can not exceed 6.0"
        # "* If `model_avg_score` is low (6.0 < `model_avg_score` < 6.5), overall score can not exceed 6.5"
        "* A drug with low safety (Safety score < 5.0) can't be positively recommended."
    )
    overall_prioritization_score: float = Field(
        default=0.0,
        ge=0.0, 
        le=10.0, 
        description="Overall prioritization score (0.0 to 10.0). Weighted composite score: 10% Safety + 10% PK + 10% PD + 70% ModelAvg + boosts - penalties. " \
        # "MUST be penalized or capped if model_avg_score is  low (< 6.5). A candidate cannot score > 7.0 if model_avg_score is low, or if it is unsafe (Safety score < 5.0)."
    )



In [ ]:
### evaluator.py

import instructor
from openai import OpenAI
import re
# from schemas import ADMETEvaluation
from typing import Dict, Set

# CONFIGURATION: local Ollama server (keeps DrugBank text on this machine)
client = instructor.from_openai(
    OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"),  # OpenAI-compatible client pointed at Ollama
    mode=instructor.Mode.JSON
)
MODEL_NAME = "qwen2.5:7b"

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Pre-Processing: Group Raw Keys in Python to simplify the LLM task
# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Target Normalization & Noise Patterns
# ---------------------------------------------------------------------------

# Verbose DrugBank entity prefixes mapped to standard scientific acronyms
TARGET_NORMALIZATIONS: Dict[str, str] = {
    # Phase I (Cytochromes)
    r"^Cytochrome P-450\s+": "",
    r"^Cytochrome P-450 Enzyme": "CYP",
    # Phase II Metabolism Enzymes
    r"^UDP-Glucuronosyltransferase\s*": "UGT",
    r"^Glutathione S-Transferase\s*": "GST",
    r"^Sulfotransferase\s*": "SULT",
    r"^N-acetyltransferase\s*": "NAT",
    r"^Thiopurine S-methyltransferase": "TPMT",
    r"^Carboxylesterase\s*": "CES",
    # Transporters (ABC & SLC Families)
    r"^Organic Anion Transporting Polypeptide\s*": "OATP",
    r"^Organic Anion Transporter\s*": "OAT",
    r"^Organic Cation Transporter\s*": "OCT",
    r"^Multidrug Resistance-Associated Protein\s*": "MRP",
    r"^Breast Cancer Resistance Protein": "BCRP",
    r"^Bile Salt Export Pump": "BSEP",
    r"^P-glycoprotein\s*": "P-gp ",
    # Receptors & Enzymes
    r"^Monoamine Oxidase\s*": "MAO-",
}

# Formula for ADME interaction tags: "<Target> <Role> [with a Narrow Therapeutic Index]"
ADME_ROLE_PATTERN = re.compile(
    r"^(.*?)\s+(Substrates|Inhibitors|Inducers|Modulators)(?:\s+with\s+a\s+Narrow\s+Therapeutic\s+Index)?$",
    re.IGNORECASE,
)

# Broad structural/chemical ontology noise (zero ADMET or clinical scoring signal)
CHEMICAL_NOISE_PATTERNS = re.compile(
    r"^(amino acids|proteins|albumins|peptides|lipids|heterocyclic|benzene|"
    r"carboxylic|diterpenes|cycloparaffins|cyclodecanes|terpenes|small molecules|"
    r"organic compounds|macromolecular substances)",
    re.IGNORECASE,
)


# ---------------------------------------------------------------------------
# Normalization & Deduplication Functions
# ---------------------------------------------------------------------------


def normalize_target_name(target_name: str) -> str:
    """Applies normalization rules to convert verbose DrugBank target names into standard acronyms."""
    for pattern, replacement in TARGET_NORMALIZATIONS.items():
        target_name = re.sub(
            pattern, replacement, target_name, flags=re.IGNORECASE
        )
    return target_name.strip()


def clean_and_deduplicate_categories(raw_categories_str: str) -> str:
    """Universal cleaner for DrugBank category tags across all drugs.

    Dynamically collapses ADME roles (CYPs, Phase II enzymes, ABC/SLC
    transporters) and removes chemical ontology noise without requiring
    hardcoded drug tags.
    """
    if not raw_categories_str or raw_categories_str in ("UNKNOWN", "None"):
        return "None specified"

    tags = [t.strip() for t in raw_categories_str.split("||") if t.strip()]

    adme_map: Dict[str, Set[str]] = (
        {}
    )  # e.g., {'CYP3A4': {'Substrate', 'Inducer', 'NTI'}}
    filtered_clinical_tags: Set[str] = set()
    has_global_nti = False

    for tag in tags:
        tag_lower = tag.lower()

        # 1. Capture global Narrow Therapeutic Index flag
        if "narrow therapeutic index" in tag_lower:
            has_global_nti = True

        # 2. Check if tag follows ADME role formula: "<Target> <Role> [NTI]"
        adme_match = ADME_ROLE_PATTERN.match(tag)
        if adme_match:
            raw_target = adme_match.group(1).strip()
            role = adme_match.group(2).capitalize()

            # Normalize verbose target names to clean acronyms
            target_name = normalize_target_name(raw_target)

            if target_name not in adme_map:
                adme_map[target_name] = set()

            adme_map[target_name].add(role)
            if "narrow therapeutic index" in tag_lower:
                adme_map[target_name].add("NTI")
            continue

        # 3. Filter out broad chemical ontology noise
        if CHEMICAL_NOISE_PATTERNS.search(tag_lower):
            continue

        # 4. Retain core clinical, safety, and pharmacological classes
        filtered_clinical_tags.add(tag)

    # 5. Deduplicate nested/overlapping clinical tags (keep longer/more specific terms)
    sorted_clinical = sorted(filtered_clinical_tags, key=len, reverse=True)
    deduped_clinical = []
    for tag in sorted_clinical:
        if not any(
            tag != existing and tag in existing for existing in deduped_clinical
        ):
            deduped_clinical.append(tag)

    # 6. Assemble consolidated string
    consolidated = []

    if has_global_nti:
        consolidated.append("Narrow Therapeutic Index (NTI)")

    # Add clinical/pharmacological terms
    consolidated.extend(sorted(deduped_clinical))

    # Add dynamically mapped ADME targets
    for target, roles in sorted(adme_map.items()):
        role_order = ["Substrate", "Inhibitor", "Inducer", "Modulator"]
        active_roles = [r for r in role_order if r in roles]
        role_str = "/".join(active_roles) if active_roles else "Modulator"
        if "NTI" in roles:
            role_str += " - NTI"
        consolidated.append(f"{target} {role_str}")

    return ", ".join(consolidated) if consolidated else "None specified"

def pregroup_drugbank_data(drug_name: str, clean_data: dict) -> dict:
    """Combines fragmented DrugBank dictionary keys into 5 structured text blocks."""

    def get_val(key):
        return str(clean_data.get(key, "UNKNOWN")).strip()

    # 1. Background
    clinical = f"Description: {get_val('description')}\nIndication: {get_val('indication')}"

    # 2. PD & Targets
    pd = (
        f"Mechanism of Action: {get_val('mechanism-of-action')}\n"
        f"Pharmacodynamics: {get_val('pharmacodynamics')}\n"
        f"Targets: {get_val('targets')}\n"
        f"Affected Organisms: {get_val('affected-organisms')}"
    )

    # 3. PK / ADME (Consolidate all separate PK keys)
    pk = (
        f"Absorption: {get_val('absorption')}\n"
        f"Protein Binding: {get_val('protein_binding')}\n"
        f"Volume of Distribution: {get_val('volume_of_distribution')}\n"
        f"Metabolism: {get_val('metabolism')}\n"
        f"Half-Life: {get_val('half_life')}\n"
        f"Clearance: {get_val('clearance')}\n"
        f"Route of Elimination: {get_val('route_of_elimination')}"
    )

  # 4. Safety
    safety = (
        f"Toxicity: {get_val('toxicity')}"#\nCategories: {get_val('categories')}"
    )

    # 5. Dosages (Extract routes/forms using regex to avoid 150+ repetitive lines)
    raw_dosages = get_val("dosages")
    routes = set(re.findall(r"Route:\s*([^,|]+)", raw_dosages, re.IGNORECASE))
    forms = set(re.findall(r"Form:\s*([^,|]+)", raw_dosages, re.IGNORECASE))

    dosage_str = (
        f"Routes: {', '.join(routes) if routes else 'Not specified'}\n"
        f"Forms: {', '.join(forms) if forms else 'Not specified'}"
    )

    # 0. Categories as a formatted string
    raw_categories = get_val("categories")
    category_tags_str = clean_and_deduplicate_categories(raw_categories)
    # if raw_categories:
    #     tags_list = [t.strip() for t in raw_categories.split("||") if t.strip()]
    #     category_tags_str = ", ".join(tags_list) if tags_list else "None specified"
    # else:
    #     category_tags_str = "None specified"

    # Safely extract ModelAvg as a float or None
    raw_avg = clean_data.get("ModelAvg", None)
    try:
        model_avg = float(raw_avg) / 10 if raw_avg is not None and raw_avg != "" else None
    except (ValueError, TypeError):
        model_avg = None

    return {
        "category_tags": category_tags_str,
        "clinical_overview_raw": clinical,
        "pharmacodynamics_raw": pd,
        "pk_properties_raw": pk,
        "safety_raw": safety,
        "dosage_raw": dosage_str,
        "model_avg_raw": model_avg,
    }

def extract_clean_admet(drug_name: str, clean_data: dict) -> ADMETSummary:
    """Agent 1: Distills raw text categories into a structured ADMETSummary."""

    grouped = pregroup_drugbank_data(drug_name, clean_data)
    # print(grouped)

    system_prompt = f"""
    You are an expert pharmaceutical data curator and toxicologist. Your task is to convert raw DrugBank dictionary fields into a clean, canonical ADMETSummary object. Extract clear, factual ADMET statements. Omit duplicate text, formatting artifacts, and irrelevant administrative details (e.g. who and how initially produced the drug).

    Set `drug_name` explicitly to "{drug_name}".
    Set `model_avg_score` to the provided ModelAvg {grouped['model_avg_raw']} float value.
    Set `category_tags` to the provided array of category tag keywords {grouped['category_tags']}.

    STRICT RULES:
    1. CATEGORY TAGS USE: A string of high-level ontology CATEGORY TAGS is provided. Use these tags actively across ALL 5 summary fields whenever they provide relevant additional information (e.g., therapeutic class, MOA, metabolic pathways, toxicities, or administration methods).
    2. EXCLUSIVE SOURCE OF TRUTH: You MUST rely EXCLUSIVELY on the provided raw text and category tags. 
    3. NO PRE-TRAINED KNOWLEDGE: Do NOT output disease indications (e.g., Multiple Sclerosis), specific enzymes (e.g., CYP3A4, CYP2C8), binding properties, or half-lives unless the EXACT words or numbers appear in the input text.
    4. HANDLING UNKNOWN SECTIONS:
        - If a section's raw text is "Mechanism of Action: UNKNOWN", "Toxicity: UNKNOWN", etc., your output MUST NOT invent specific biological mechanisms or toxicities.
        - You HAVE TO reference category tags relevant to that field (e.g., mention "Immunosuppressive Agents" under clinical/PD). 
        - If both the raw text is UNKNOWN and no relevant tags exist for that field, output "UNKNOWN" (or ["UNKNOWN"] for lists).

    ======================================================================
    FEW-SHOT EXEMPLAR (FOLLOW THIS PATTERN)
    ======================================================================
    [EXAMPLE INPUT]
    CANDIDATE: DRUG_NAME
    COMPUTATIONAL PREDICTED INTERACTION PROBABILITY (ModelAvg): 8.15

    0. CATEGORY TAGS: ["Kinase Inhibitors", "Cardiotoxic Agents", "Antineoplastic Agents"]
    1. CLINICAL DATA:
    Description: UNKNOWN
    Indication: UNKNOWN
    2. PHARMACODYNAMICS DATA:
    Mechanism of Action: Reversible inhibition of tyrosine kinase signaling pathways.
    Pharmacodynamics: UNKNOWN
    Targets: VEGFR2 (KDR)
    Affected Organisms: UNKNOWN
    3. PHARMACOKINETICS DATA:
    Absorption: UNKNOWN
    Protein Binding: UNKNOWN
    Volume of Distribution: UNKNOWN
    Metabolism: UNKNOWN
    Half-Life: UNKNOWN
    Clearance: UNKNOWN
    Route of Elimination: UNKNOWN
    4. SAFETY & TOXICITY DATA:
    Toxicity: UNKNOWN
    5. DOSAGE & ROUTE DATA:
    Routes: Intravenous
    Forms: Solution for Injection, 10 mg/mL

    [EXAMPLE CORRECT OUTPUT]
    {{
    "drug_name": "DRUG_NAME",
    "model_avg_score": 8.15,
    "category_tags": ["Kinase Inhibitors", "Cardiotoxic Agents", "Antineoplastic Agents"],
    "clinical_overview": "Antineoplastic cardiotoxic agent acting as kinase inhibitor. Specific clinical indications UNKNOWN.",
    "pharmacodynamics_and_targets": "Reversible inhibitor of VEGFR2 (KDR) tyrosine kinase signaling.",
    "pk_properties": "UNKNOWN. Specific absorption, PPB, volume of distribution, metabolic pathways (CYP enzymes), half-life, and clearance parameters are unspecified in raw text.",
    "toxicity_highlights": [
        "Antineoplastic Agents",
        "Classified under Cardiotoxic Agents category tag"
    ],
    "dosage_and_route_summary": [
        "Intravenous",
        "Solution for Injection"
    ]
    }}
    ======================================================================
    FEW-SHOT EXEMPLAR (FINISH LINE)
    ======================================================================

    Field Mapping Rules:
    1. Clinical Overview: Synthesize 'description', 'indication', and therapeutic category tags from 'category_tags' into 1-2 concise sentences.

    2. Pharmacodynamics & Targets:
        - Synthesize 'targets', 'mechanism-of-action', 'pharmacodynamics', and 'affected-organisms' and 'category_tags'.
        - Use any relevant tags from 'category_tags', such as enzyme/receptor inhibitor/agonist tags (e.g., "DNA Polymerase Inhibitor", "Nucleic Acid Synthesis Inhibitors").
        - Note specific biological targets (e.g., Topoisomerase II, HER2; and whether the drug is an inhibitor, agonist, or antagonist), MoA, and target organisms/pathogens (e.g., host vs. viral/bacterial organism spectrum).

    3. Pharmacokinetics (ADME):
        - Synthesize 'absorption', 'protein_binding', 'volume_of_distribution', 'metabolism', 'half_life', 'clearance', 'route_of_elimination' and 'category_tags'
        - Use any relevant tags from 'category_tags' indicating metabolic pathways (e.g., "CYP1A2 Substrates"), transporters (e.g., "OAT1 inhibitors", "MATE 1 Substrates"), or excretion (e.g., "Renally Excreted")'.
        - Note specific Plasma Protein Binding (PPB %), Volume of Distribution (Vd), Clearance (CL), primary CYP metabolizing enzymes, and excretion pathways (renal vs. hepatic).

    4. Toxicity Highlights:
        - Extract major organ toxicities and safety warnings from 'toxicity' and 'category_tags'.
        - Use any relevant tags from 'category_tags' (e.g., "Nephrotoxic agents", "Hepatotoxic", "Cardiotoxic", "Carcinogens").

    5. Dosage & Route Summary:
        - The 'dosages' field contains raw, repetitive commercial listings. DO NOT copy individual strength values.
        - Distill the listings into clean bullets detailing primary delivery routes (e.g., Oral tablet, IV infusion, Subcutaneous), include also any relevant route-specific category tags from 'category_tags (e.g., "Inhalation Agents", "Oral Solutions").
    """

    # prompt = f"### DRUG CANDIDATE: {drug_name}\n\n"
    # for k, v in clean_data.items():
    #     prompt += f"#### {k.upper()}:\n{v}\n\n"

    prompt = (
        f"### CANDIDATE: {drug_name}\n\n"
        f"COMPUTATIONAL PREDICTED INTERACTION PROBABILITY (ModelAvg): {grouped['model_avg_raw']}\n\n"
        f"0. CATEGORY TAGS:\n{grouped['category_tags']}\n\n"
        f"1. CLINICAL DATA:\n{grouped['clinical_overview_raw']}\n\n"
        f"2. PHARMACODYNAMICS DATA:\n{grouped['pharmacodynamics_raw']}\n\n"
        f"3. PHARMACOKINETICS DATA:\n{grouped['pk_properties_raw']}\n\n"
        f"4. SAFETY & TOXICITY DATA:\n{grouped['safety_raw']}\n\n"
        f"5. DOSAGE & ROUTE DATA:\n{grouped['dosage_raw']}\n"
    )

    summary = client.chat.completions.create(
        model=MODEL_NAME,
        response_model=ADMETSummary,
        messages=[
            {"role": "system", "content": system_prompt,},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        timeout=180,
        max_retries=4,
        max_tokens= 4096,  # OpenAI client cap
        extra_body={
            "options": {
                "num_ctx": 8192,     # Expands input context window to fit full drug text
                "num_predict": 4096   # Sets output token budget
            }
        }
    )

    # summary.drug_name = drug_name
    if not summary.drug_name:
        summary.drug_name = drug_name
    if grouped["model_avg_raw"] is not None:
        summary.model_avg_score = grouped["model_avg_raw"]

    return summary

def evaluate_from_summary(summary: ADMETSummary) -> ADMETEvaluation:
    """Agent 2: Scores the candidate based purely on the distilled ADMETSummary, returning an ADMETEvaluation"""

    SYSTEM_PROMPT = """
    You are an expert pharmacologist, toxicologist, and computational biologist evaluating drug candidates for repurposing based on ADMET (Absorption, Distribution, Metabolism, Excretion, Toxicity) profiling.

    Your task is to analyze the provided canonical ADMETSummary object and issue sub-scores (0.0 to 10.0) across the specified core dimensions, followed by an overall prioritization score to choose from a list of drugs.

    ### EVALUATION CRITERIA:

    1. Boolean Classification Flags:
        - vaccine: Is this candidate a vaccine?
        - antibody: Is this candidate a monoclonal antibody or biologic antibody?
        - chemotherapy: Is this a chemotherapy agent?
        - cytotoxic: Does it exert direct cellular cytotoxicity?
        - immunosuppressor: Is it an immunosuppressive agent?
        - antiinflammatory: Does it exert anti-inflammatory action?
        - IV: Is it ONLY available via intravenous route?
        - Oral_Inhalation: Is the drug available via oral or inhalation routes?
        - antiviral: Is it used for treating viral infections or targeting viral mechanisms?
        - DNA_virus: Does it target DNA viruses specifically?
        - RNA_virus: Does it target RNA viruses specifically?
        - endo: Does it target endothelial dysfunction/inflammation?
        - respiratory: Does it target respiratory dysfunction?
        - neuronal: Does it target neuronal dysfunction or act as a neuroprotective agent?

    2. Quantitative Scoring (0.0 to 10.0):
        - General scoring rules:
            * Write your step-by-step Chain-of-Thought rationale BEFORE assigning each numerical score.
            * Absence of evidence is NOT evidence of good performance. Unspecified data represents clinical risk.
            * If `pk_properties`, 'pharmacodynamics_and_targets' or 'toxicity_highlights' is 'UNKNOWN':
                i) Write 'UNKNOWN' for the corresponding missing entry rationale.
                ii) set their score to 0.0
                iii) You MUST add 'Missing PK data', 'Missing PD data', or 'Missing Safety data', to `missing_entries`, for any that are 'UNKNOWN'.

        A. Pharmacodynamics Score (0.0 - 10.0): Based on 'pharmacodynamics_and_targets', . High scores require clear molecular targets, strong binding affinity/selectivity, and defined therapeutic pathways.

        B. Pharmacokinetics Score (0.0 - 10.0): Based on 'pk_properties'. Evaluate whether oral bioavailability, plasma protein binding, volume of distribution (Vd), clearance rate, half-life, and metabolic stability (CYP metabolic interactions) are favorable.

        C. Safety Score (0.0 - 10.0): Based on 'toxicity_highlights'. Deduct points heavily for acute/chronic toxicity, organ toxicities (e.g. cardiotoxicity, hepatotoxicity), myelosuppression, narrow therapeutic index, mutagenicity, or black box warnings.

    3. List Summaries:
        - admin_routes: Based on 'dosage_and_route_summary' provide a list of the available routes of administration (e.g., Oral, Topical, IV).
        - targeted_viruses: List of viruses target by drug, if drug is antiviral
        - major_red_flags: Critical safety, toxicity, or interaction concerns, if any.
        - key_advantages: Key favorable traits (e.g., long half-life, high oral bioavailability, simple oral dosing or inhalation available).
        - missing_entries: List of missing entries (PD, PK, or Safety), if any are missing.

    4. OVERALL PRIORITIZATION:
        A. Model Avg Score Constraints:
        - model_avg_score: Simply return 'model_avg_score'. `model_avg_score` represents the prior computational model's interaction probability (0.0 to 10.0). 
        - `overall_prioritization_score` MUST explicitly incorporate `model_avg_score`.

        B. Disqualifiers, Penalties and Boosters:
            * A drug should be HEAVILY penalized (- 2.5 points) for repurposing if it is a vaccine, antibody, chemotherapy, or cytotoxic agent.
            * A drug should be penalized (-0.5 to -1.0 point to overall score)) if it is an immunosuppresor, antiinflammatory drug or if it is ONLY available for administration through intravenous (IV) route.
            * A drug is favored (MINOR BOOST: +0.5 to +1.0 point to overall score) if it can be administered orally or by inhalation or if it inhibits a DNA_virus.
            * A drug is high-priority (MAJOR BOOST: +1.5 to +2.0 points to overall score) for repurposing if it inhibits an RNA_virus; or if the drug is TRUE for 'endo', 'respiratory' or 'neuronal' flags.  

        C. overall_prioritization_rationale:
            * Base score = 'model_avg_score' * .7 + 'pharmacokinetics_score' * .1 + 'safety_score' * .1 + 'pharmacodynamics_score' * .1.
            * overall_prioritization_score (0.0 to 10.0) = 'Base score' + boosts - penalties.
            * Provide a concise 2-3 sentence summary taking into account the above considerations. Specifcally mention any high-priority/favorable boolean flags boosting its overall score and penalties/red_flags decreasing its overall score.
            * If a drug has very low safety ('safety_score' < 4.0) but good overall score (>6.0), explain why in the rationale. 
    """
            # * MUST write overall_prioritization_rationale FIRST before outputting `overall_prioritization_score`. Mention how `model_avg_score` impacts the final score.
            # * A high Overall Score (>7.0) REQUIRES both high Pharmacodynamics/Efficacy AND acceptable Safety. Good PK or Dosage alone cannot salvage a highly toxic or ineffective drug.
        # - HARD CAP: If `model_avg_score` is weak (< 6.0), `overall_prioritization_score` CANNOT exceed 6.0 regardless of how clean the drug's safety or PK profiles are.
        # - If `model_avg_score` is low (6.0 < `model_avg_score` < 6.5), `overall_prioritization_score` MUST be scaled below 6.5.
            # * A drug with a high Overall Score (>7.0) for prioritization must have a high 'model_avg_score'(>7.0).


    # Format the input dictionary into a clean text prompt
    # text_block = f"### DRUG CANDIDATE: {drug_name}\n\n"
    # for category, content in text_categories.items():
    #     text_block += f"#### {category.upper()}:\n{content}\n\n"

    # user_prompt = f"{text_block}\nEvaluate the ADMET profile above, classify the drug according to the system prompt rules, and generate the structured scoring object."
    user_prompt = (f"Evaluate this clean profile:\n{summary.model_dump_json(indent=2)}")

    response = client.chat.completions.create(
        model=MODEL_NAME,
        response_model=ADMETEvaluation,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0,
        timeout=180,
        max_retries=4,
        max_tokens= 4096,  # OpenAI client cap
        extra_body={
            "options": {
                "num_ctx": 8192,     # Expands input context window to fit full drug text
                "num_predict": 4096   # Sets output token budget
            }
        }
    )
    
    return response

In [ ]:
### main.py

import json
import os
import time
from tqdm import tqdm
# from evaluator import evaluate_single_drug

# INPUT_FILE = "drugs_admet_text.json"
# CHECKPOINT_FILE = "admet_evaluations_checkpoint.json"

def load_input_data(INPUT_FILE) -> dict:
    """Loads the input dictionary of drugs and their text categories."""
    with open(INPUT_FILE, "r") as f:
        return json.load(f)

def run_pipeline(INPUT_FILE, CHECKPOINT_FILE =  "admet_evaluations_checkpoint.json"):
    drugs_data = load_input_data(INPUT_FILE)
    print(f"Loaded {len(drugs_data)} drugs for ADMET evaluation.")

    # Load existing checkpoint if available
    evaluations = {}
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            evaluations = json.load(f)
        print(f"Resuming from checkpoint: {len(evaluations)} drugs already evaluated.")

    # Process remaining drugs
    # for drug_name, text_categories in tqdm(drugs_data.items(), desc="Evaluating Drugs"):
    for drug_entry in tqdm(drugs_data, desc='Evaluating Drugs'):
        drug_name = drug_entry.get('name')
        if drug_name in evaluations:
            continue  # Skip already processed drugs

        text_categories = {
                k: str(v)
                for k, v in drug_entry.items()
                if k not in ['name', 'drugbank_id', 'fda-label', 'safety-sheet'] and v
            }

        try:
            # result = evaluate_single_drug(drug_name, text_categories)

            # Step 1: Agent 1 - Extraction & Synthesis
            admet_summary = extract_clean_admet(drug_name, text_categories)

            # Step 2: Agent 2 - Evaluation & Scoring
            result = evaluate_from_summary(admet_summary)

            evaluations[drug_name] = result.model_dump()

            # Save checkpoint after each item
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(evaluations, f, indent=4)

        except Exception as e:
            print(f"\n[Error processing {drug_name}]: {e}")
            time.sleep(1)

    print("\nPipeline execution complete! All results saved to checkpoint.")

# if __name__ == "__main__":
#     run_pipeline()

In [ ]:
### rank_and_export.py

import json
import pandas as pd

# CHECKPOINT_FILE = "admet_evaluations_checkpoint.json"
# OUTPUT_CSV = "prioritized_admet_drugs.csv"

def generate_report(OUTPUT_CSV, CHECKPOINT_FILE =  "admet_evaluations_checkpoint.json"):
    with open(CHECKPOINT_FILE, "r") as f:
        data = json.load(f)

    records = []
    for drug_name, eval_data in data.items():
        records.append({
            "Drug Name": drug_name,
            "Overall Score": eval_data["overall_prioritization_score"],
            "Safety Score": eval_data["safety_score"],
            "PK Score": eval_data["pharmacokinetics_score"],
            "PD Score": eval_data["pharmacodynamics_score"],
            "Red Flag Count": len(eval_data["major_red_flags"]),
            "Major Red Flags": "; ".join(eval_data["major_red_flags"]),
            "Key Advantages": "; ".join(eval_data["key_advantages"]),
            "Recommendation Rationale": eval_data["overall_prioritization_rationale"]
        })

    df = pd.DataFrame(records)

    # Sort by Overall Score descending, then Safety Score descending
    df = df.sort_values(by=["Overall Score", "Safety Score"], ascending=[False, False])

    # Save to CSV
    df.to_csv(OUTPUT_CSV, index=False, sep='\t')
    print(f"Successfully generated prioritized report: {OUTPUT_CSV}")

    # Display Top 10 Candidates
    print("\nTop 10 Prioritized Drug Candidates:")
    print(df[["Drug Name", "Overall Score", "Safety Score", "Red Flag Count"]].head(10).to_string(index=False))

# if __name__ == "__main__":
#     generate_report()

In [ ]:
adams = pd.read_csv("~/.../Adam's copy of Prel Results - priorities.csv")
set(adams['drug_name'][:10].dropna().str.strip().str.lower())

In [ ]:
adams['ModelAvg'] = adams[['endos', 'iAstro', 'iNeuron', 'AT']].mean(axis=1).round(2)
adams

In [ ]:
adams.columns

In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

def parse_drugbank_xml(file_path):
    # DrugBank uses a namespace, we must account for it
    # Note: If your XML doesn't have the 'db:' prefix, 
    # you can remove 'ns' and the prefixes in findall.
    ns = {'db': 'http://www.drugbank.ca'}
    
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    drug_data = []

    # Iterate through each drug entry
    for drug in root.findall('db:drug', ns):
        # 1. Extract basic flat fields
        row = {
            'drugbank_id': drug.findtext('db:drugbank-id[@primary="true"]', namespaces=ns),
            'name': drug.findtext('db:name', namespaces=ns),
            'type': drug.get('type'),
            # 'state': drug.findtext('db:state', namespaces=ns),
            # 'atc': drug.findtext('db:atc-codes', namespaces=ns),
            'description': drug.findtext('db:description', namespaces=ns),
            'indication': drug.findtext('db:indication', namespaces=ns),
            'pharmacodynamics': drug.findtext('db:pharmacodynamics', namespaces=ns),
            'toxicity': drug.findtext('db:toxicity', namespaces=ns),
            'fda-label': drug.findtext('db:fda-label', namespaces=ns),
            'safety-sheet': drug.findtext('db:msds', namespaces=ns),
            'mechanism-of-action': drug.findtext('db:mechanism-of-action', namespaces=ns),
            'metabolism': drug.findtext('db:metabolism', namespaces=ns),
            'absorption': drug.findtext('db:absorption', namespaces=ns),
            'half_life': drug.findtext('db:half-life', namespaces=ns), # Elimination/Excretion
            'protein_binding': drug.findtext('db:protein-binding', namespaces=ns), # Distribution
            'clearance': drug.findtext('db:clearance', namespaces=ns), # Elimination/Excretion
            'volume_of_distribution': drug.findtext('db:volume-of-distribution', namespaces=ns), # distribution
            'route_of_elimination': drug.findtext('db:route-of-elimination', namespaces=ns), # Elimination/Excretion
        }
        
        # 2. Extract and flatten the 'groups' (the fix for your \n issue)
        group_elements = drug.findall('db:groups/db:group', ns)
        row['groups'] = "|".join([g.text for g in group_elements if g.text])
        
        group_elements = drug.findall('db:synonyms/db:synonym', ns)
        row['synonyms'] = "|".join([g.text for g in group_elements if g.text])
        
        group_elements = drug.findall('db:affected-organisms/db:affected-organism', ns)
        row['affected-organisms'] = "|".join([g.text for g in group_elements if g.text])
        
        group_elements = drug.findall('db:targets/db:target/db:name', ns)
        row['targets'] = "|".join([g.text for g in group_elements if g.text])
        
        # Schema layout: <atc-codes> -> <atc-code code="J05AB01">
        atc_elements = drug.findall('db:atc-codes/db:atc-code', ns)
        # Extract the 'code' attribute from each matching xml element
        # atc_list = [elem.get('code') for elem in atc_elements if elem.get('code')]
        # # Flatten them into a single pipe-separated string (e.g., "J05AB01|J05AB02")
        # row['atc'] = "|".join(atc_list) if atc_list else "None"

        atc_list = []
        atc_labels = []
        for elem in atc_elements:
            code_attr = elem.get('code')
            if code_attr:
                atc_list.append(code_attr)
                
                # Find all nested <level> elements inside this specific ATC block
                levels = elem.findall('db:level', ns)
                if levels:
                    # The first <level> tag in the XML represents the most specific category (e.g., J05AF)
                    highest_level_text = levels[0].text
                    if highest_level_text:
                        # .strip() removes newline characters (\n) and surrounding whitespace
                        atc_labels.append(highest_level_text.strip())

        # 2. Join strings with pipes for your dataset columns
        row['atc'] = "|".join(atc_list) if atc_list else "None"
        row['atc_names'] = " || ".join(atc_labels) if atc_labels else "None"

        # 3. Optional: Extract other nested fields like Categories
        categories = drug.findall('db:categories/db:category/db:category', ns)
        row['categories'] = " || ".join([c.text for c in categories if c.text])

        # categories = drug.findall('db:therapeutic-categories/db:therapeutic-category/db:therapeutic-category', ns)
        # row['therapeutic-categories'] = "|".join([c.text for c in categories if c.text])

        # 1. Find all reaction elements within the snp-adverse-drug-reactions tag
        reaction_elements = drug.findall('db:snp-adverse-drug-reactions/db:reaction', ns)

        reaction_list = []

        for reaction in reaction_elements:
            gene_elem = reaction.find('db:gene-symbol', ns)
            desc_elem = reaction.find('db:description', ns)
            
            # Ensure both elements exist and have valid text
            if gene_elem is not None and desc_elem is not None:
                gene = gene_elem.text.strip() if gene_elem.text else ""
                desc = desc_elem.text.strip() if desc_elem.text else ""
                
                if gene and desc:
                    # Format exactly as requested
                    formatted_reaction = f"{gene}: {desc}"
                    reaction_list.append(formatted_reaction)

        # 2. Join multiple entries with a newline (or pipes '|') to keep the cell organized
        row['snp_adverse_reactions'] = "\n".join(reaction_list) if reaction_list else "None"

        # Extract SMILES structure
        row['smiles'] = ''
        calc_props = drug.find('db:calculated-properties', ns)
        if calc_props is not None:
          for prop in calc_props.findall('db:property', ns):
            kind = prop.find('db:kind', ns)
            if kind is not None and kind.text == 'SMILES':
              val = prop.find('db:value', ns)
              if val is not None:
                row['smiles'] = val.text
                break

        # Extract Structured Dosages (Route, Form, Strength)
        dosage_list = []
        dosages_node = drug.find('db:dosages', ns)

        if dosages_node is not None:
            for dosage in dosages_node.findall('db:dosage', ns):
                form_elem = dosage.find('db:form', ns)
                route_elem = dosage.find('db:route', ns)
                strength_elem = dosage.find('db:strength', ns)

                parts = []

                if form_elem is not None and form_elem.text and form_elem.text.strip():
                    parts.append(f'Form: {form_elem.text.strip()}')

                if route_elem is not None and route_elem.text and route_elem.text.strip():
                    parts.append(f'Route: {route_elem.text.strip()}')

                if strength_elem is not None and strength_elem.text and strength_elem.text.strip():
                    parts.append(f'Strength: {strength_elem.text.strip()}')

                # Join non-None parts for this specific dosage entry
                if parts:
                    dosage_list.append(', '.join(parts))

        # Combine all distinct dosage entries for the drug into a single string
        row['dosages'] = ' || '.join(dosage_list)

        drug_data.append(row)

        print(f"Processed drug: {row['name']}")

    return pd.DataFrame(drug_data)

# Usage
DB = parse_drugbank_xml('<path/to/drugbank_full_database/full database.xml>')
DB

In [ ]:
DB.columns

In [ ]:
DB[['name', 'description', 'indication', 'pharmacodynamics', 'toxicity', 'mechanism-of-action', 'metabolism', 'absorption', 'half_life', 'protein_binding', 'clearance', 'volume_of_distribution', 'route_of_elimination', 'affected-organisms', 'targets', 'categories', 'snp_adverse_reactions', 'dosages']]

In [ ]:
adams['ModelAvg'] = adams[['endos', 'iAstro', 'iNeuron', 'AT']].apply(pd.to_numeric, errors='coerce').mean(axis=1).round(2)

model_avg_map = dict(
    zip(
        adams['drug_name'].dropna().astype(str).str.strip().str.lower(),
        adams['ModelAvg']
    )
)

target_names = set(adams['drug_name'][:9].dropna().str.strip().str.lower()) | {'acyclovir', 'everolimus', 'Berberine', 'monomethyl fumarate', 'tetrachlorodecaoxide', 'imiquimod', 'sirolimus', 'papain', 'belatacept', 'alefacept', 'tretinoin', 'colchicine'}
cols = ['name', 'description', 'indication', 'pharmacodynamics', 'toxicity', 'mechanism-of-action', 'metabolism', 'absorption', 'half_life', 'protein_binding', 'clearance', 'volume_of_distribution', 'route_of_elimination', 'affected-organisms', 'targets', 'categories', 'dosages']
filtered_db = DB[
    DB['name'].fillna('').astype(str).str.strip().str.lower().isin(target_names)
][cols].copy()

filtered_db['ModelAvg'] = (
    filtered_db['name']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(model_avg_map)
)

filtered_db

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def generate_filtered_db_txt_summary(
    df: pd.DataFrame, output_path: Path
) -> None:
    """Generates a formatted .txt summary from filtered_db DataFrame ordered by

    ModelAvg descending, grouped into Agent 1 schema sections.
    """
    # Create a copy and sort by ModelAvg descending (NaNs placed at the end)
    df_sorted = df.copy()
    if "ModelAvg" in df_sorted.columns:
        df_sorted = df_sorted.sort_values(by="ModelAvg", ascending=False)

    lines = []

    # Map sections and corresponding DataFrame columns
    sections = [
        (
            "0. CATEGORY TAGS",
            [
                ("Categories", "categories"),
            ],
        ),
        (
            "1. CLINICAL DATA",
            [
                ("Description", "description"),
                ("Indication", "indication"),
            ],
        ),
        (
            "2. PHARMACODYNAMICS DATA",
            [
                ("Mechanism of Action", "mechanism-of-action"),
                ("Pharmacodynamics", "pharmacodynamics"),
                ("Targets", "targets"),
                ("Affected Organisms", "affected-organisms"),
            ],
        ),
        (
            "3. PHARMACOKINETICS DATA",
            [
                ("Absorption", "absorption"),
                ("Protein Binding", "protein_binding"),
                ("Volume of Distribution", "volume_of_distribution"),
                ("Metabolism", "metabolism"),
                ("Half-Life", "half_life"),
                ("Clearance", "clearance"),
                ("Route of Elimination", "route_of_elimination"),
            ],
        ),
        (
            "4. SAFETY & TOXICITY DATA",
            [
                ("Toxicity", "toxicity"),
            ],
        ),
        (
            "5. DOSAGE & ROUTE DATA",
            [
                ("Dosages", "dosages"),
            ],
        ),
    ]

    for _, row in df_sorted.iterrows():
        drug_name = row.get("name", "Unknown")
        model_avg = row.get("ModelAvg")

        model_avg_str = (
            f"{float(model_avg):.3f}"
            if pd.notna(model_avg) and model_avg is not None
            else "N/A"
        )

        lines.append("=" * 90)
        lines.append(f"DRUG: {drug_name} (Model Avg Score: {model_avg_str})")
        lines.append("=" * 90)

        for section_title, fields in sections:
            lines.append("\n" + "-" * 40)
            lines.append(section_title)
            lines.append("-" * 40)

            for label, col_name in fields:
                lines.append(f"\n[{label}]")
                val = row.get(col_name)

                # Handle empty/NaN/None values
                if (
                    val is None
                    or pd.isna(val)
                    or val == ""
                    or (isinstance(val, (list, dict)) and len(val) == 0)
                ):
                    lines.append("  • (None provided)")
                elif isinstance(val, list):
                    for item in val:
                        lines.append(f"  • {item}")
                else:
                    # Clean up multi-line or standard string text
                    val_str = str(val).strip()
                    lines.append(f"{val_str}\n")

        lines.append("\n\n")

    # Output to .txt file
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Filtered DB text summary successfully generated at: {output_path}")


# ---------------------------------------------------------------------------
# Execution
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Example usage:
    output_txt_file = Path("../PrelResults/ADMET_eval_tests/filtered_db_summary.txt")
    generate_filtered_db_txt_summary(filtered_db, output_txt_file)
    pass

In [ ]:
filtered_db.sort_values(by='name').iloc[:,[0,-1]]

In [ ]:
import json
import pandas as pd
import re

# 1. Prepare target drug names for case-insensitive matching
adams['ModelAvg'] = adams[['endos', 'iAstro', 'iNeuron', 'AT']].apply(pd.to_numeric, errors='coerce').mean(axis=1).round(2)

model_avg_map = dict(
    zip(
        adams['drug_name'].dropna().astype(str).str.strip().str.lower(),
        adams['ModelAvg']
    )
)

target_names = set(adams['drug_name'][:9].dropna().str.strip().str.lower()) | {'acyclovir', 'everolimus', 'Berberine', 'monomethyl fumarate', 'tetrachlorodecaoxide', 'imiquimod', 'sirolimus', 'papain', 'belatacept', 'alefacept', 'tretinoin', 'colchicine'}
# target_names = set(['acyclovir', 'paclitaxel'])
cols = ['name', 'description', 'indication', 'pharmacodynamics', 'toxicity', 'mechanism-of-action', 'metabolism', 'absorption', 'half_life', 'protein_binding', 'clearance', 'volume_of_distribution', 'route_of_elimination', 'affected-organisms', 'targets', 'categories', 'dosages']

# 2. Filter DB for matching drug names
filtered_db = DB[
    DB['name'].fillna('').str.strip().str.lower().isin(target_names)
][cols].copy()

filtered_db['ModelAvg'] = (
    filtered_db['name']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(model_avg_map)
)

# 3. Clean up NaNs / 'None' strings for clean JSON serialization
# Replace pandas NaN/NaT with empty strings (or None/null if preferred)
filtered_db = filtered_db.fillna('')

# 4. Remove bracketed citations & clean whitespace
def clean_citations(val):
  if isinstance(val, str):
    # Removes patterns like [A1573, A257614, L45231], [A123], [1, 2], etc.
    val = re.sub(r'\[[A-Za-z0-9,\s]+\]', '', val)
    # Collapse double spaces created by removing bracketed text
    return re.sub(r'\s+', ' ', val).strip()
  elif isinstance(val, list):
    # Recursively clean if any columns (e.g., categories) are stored as lists
    return [clean_citations(item) for item in val]
  return val


# Apply citation removal across all selected columns
for col in filtered_db.columns:
  filtered_db[col] = filtered_db[col].apply(clean_citations)

# Convert DataFrame rows into a list of dictionaries
records = filtered_db.to_dict(orient='records')

# 4. Save to JSON
output_path = '../PrelResults/admet_input_drugs.json'
with open(output_path, 'w', encoding='utf-8') as f:
  json.dump(records, f, indent=2, ensure_ascii=False)

print(
    f'Successfully matched and exported {len(records)} drugs to {output_path}'
)

In [ ]:
output_path = '../PrelResults/admet_input_drugs.json'

In [ ]:
# 1. Load the JSON file into a variable
# json_path = 'admet_input_drugs.json'
with open(output_path, 'r', encoding='utf-8') as f:
  admet_drugs = json.load(f)

# 2. Locate Doxorubicin (case-insensitive search)
target_name = 'Tretinoin'
drug_json = next(
    (
        drug
        for drug in admet_drugs
        if str(drug.get('name', '')).strip().lower() == target_name.lower()
    ),
    None,
)

# 3. Test evaluate_single_drug
if drug_json:
  # 1. Extract the drug name
  drug_name = drug_json.get('name')

  # 2. Extract text categories (excluding 'name' and non-string metadata)
  # This loops through your dictionary and keeps string fields with content
  text_categories = {
      k: str(v)
      for k, v in drug_json.items()
      if k not in ['name', 'drugbank_id'] and v
  }
  print(f"Initial txt:")
  for k, v in text_categories.items():
      print(f"{k}:\n{v}\n" + "-"*40)

  # --- PRE-CLEANING STEP ---
#   clean_categories = sanitize_drug_data(text_categories)
# #   print(f"Sanitized dosages:\n{clean_categories.get('dosages')}\n")
#   print(f"\nSatnized txt:")
#   for k, v in clean_categories.items():
#       print(f"{k}:\n{v}\n" + "-"*40)

  # 3. Call evaluate_single_drug with both required positional arguments
  # results = evaluate_single_drug(drug_name, text_categories)
  summary = extract_clean_admet(drug_name, text_categories)
  print(f"\nSummarized text:\n{summary.model_dump_json(indent=4)}")

  results = evaluate_from_summary(summary)

  print(f"\nEvaluation Results for {drug_name}:")
  print(results.model_dump_json(indent=4))
else:
  print(f"Could not find '{target_name}'.")

In [ ]:
output_path = '../PrelResults/admet_input_drugs.json'

for i in range(50):
    print(f"Run # {i}")
    checkpoint = '../PrelResults/ADMET_eval_tests/admet_evaluations_checkpoint'+ f'{i}' +'.json'
    report = '../PrelResults/ADMET_eval_tests/admet_eval_drugs'+ f'{i}' +'.csv'
    if os.path.exists(report):
        print(f"{report} present")
        continue

    run_pipeline(output_path, checkpoint)
    generate_report(report, checkpoint)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

def plt_and_summary(DATA_DIR, iterations = []):
  # ==========================================
  # 1. Load and Combine 100 Evaluation Runs
  # ==========================================
  dfs = []

  # Load files (handles both 'admet_eval_drugs0.csv' and 'admet_eval_drugs00.csv' naming)
  if iterations:
    file_range = iterations
    sffx = f"_{iterations[0]}_{iterations[-1]}"
  else:
    # it = range(100)
    csv_files = [f for f in Path(DATA_DIR).glob("*.csv") if f.is_file()]
    num_files = len(csv_files)
    file_range = range(num_files)
    sffx = ""
  for i in file_range:
    file_path = DATA_DIR / f"admet_eval_drugs{i}.csv"
    if not file_path.exists():
      file_path = DATA_DIR / f"admet_eval_drugs{i:02d}.csv"

    if file_path.exists():
      # Tab-separated as specified in generate_report
      df = pd.read_csv(file_path, sep="\t")
      df["run_id"] = i
      dfs.append(df)
    else:
      print(f"Warning: {file_path.name} not found.")

  df_all = pd.concat(dfs, ignore_index=True)
  print(f"Loaded {len(dfs)} runs with {len(df_all)} total records.")


  # ==========================================
  # 2. Compute Stability Metrics (5 Metrics)
  # ==========================================
  target_cols = [
      "Overall Score",
      "Safety Score",
      "PK Score",
      "PD Score",
      "Red Flag Count",
  ]


  # Custom range aggregation function
  def Range(x):
    return x.max() - x.min()


  # Calculate aggregate statistics across all target columns
  summary = (
      df_all.groupby("Drug Name")[target_cols]
      .agg(["mean", "std", "min", "max", Range])
      .round(3)
  )

  # Rename the lower column level to Title Case for clean downstream indexing
  summary = summary.rename(
      columns={"mean": "Mean", "std": "Std", "min": "Min", "max": "Max"}, level=1
  )

  # Calculate Coefficient of Variation (% CV = Std / Mean * 100) for all score metrics
  score_cols = ["Overall Score", "Safety Score", "PK Score", "PD Score"]
  for col in score_cols:
    summary[(col, "CV%")] = (
        (summary[(col, "Std")] / summary[(col, "Mean")]) * 100
    ).round(2)

  # Sort summary columns for clean reading
  summary = summary.sort_index(axis=1, level=0)
  summary = summary.sort_values(by=('Overall Score', 'Mean'), ascending=False)


  print("\n" + "=" * 90)
  print("ADMET SCORE NOISE & VARIABILITY SUMMARY (100 Runs)")
  print("=" * 90)
  print(summary.to_string())

  # Save combined summary to CSV
  summary.columns = [
      f"{metric} ({stat})" for metric, stat in summary.columns
  ]
  # title = 
  summary.reset_index().to_csv(DATA_DIR / f'admet_noise_summary{sffx}.csv', sep = '\t', index = False)

  # ==========================================
  # 1. Configuration & Flags List
  # ==========================================
  bool_flags = [
      "chemotherapy",
      "vaccine",
      "antibody",
      "cytotoxic",
      "immunosuppressor",
      "antiinflammatory",
      "IV",
      "Oral_Inhalation",
      "antiviral",
      "DNA_virus",
      "RNA_virus",
      "endo",
      "respiratory",
      "neuronal",
  ]

  # Track flag values across runs: { "DrugName": { "flag_name": [True, False, True, ...] } }
  drug_flag_records = {}

  # Discover JSON files in DATA_DIR (excluding summary/aggregated output files)
  json_files = [
      f for f in Path(DATA_DIR).glob("**/*.json")
      if "summary" not in f.name.lower()
  ]

  print(f"Found {len(json_files)} JSON run files to process...")

  # ==========================================
  # 2. Extract Boolean Flags from JSON Files
  # ==========================================
  for file_path in json_files:
      try:
          with open(file_path, "r", encoding="utf-8") as f:
              data = json.load(f)

          # Handle top-level dict mapping: {"Alefacept": {...}, "Carmustine": {...}}
          if isinstance(data, dict):
              # Check if dict is keyed by drug name at the top level
              if "drug_name" not in data and "Drug Name" not in data:
                  items = data.items()
              else:
                  drug_name = data.get("drug_name") or data.get("Drug Name")
                  items = [(drug_name, data)]
          elif isinstance(data, list):
              items = []
              for item in data:
                  drug_name = item.get("drug_name") or item.get("Drug Name")
                  if drug_name:
                      items.append((drug_name, item))
          else:
              continue

          for drug_name, drug_info in items:
              if not isinstance(drug_info, dict) or not drug_name:
                  continue

              if drug_name not in drug_flag_records:
                  drug_flag_records[drug_name] = {flag: [] for flag in bool_flags}

              # Map keys flexibly to handle potential case variations (e.g. Oral_Inhalation vs oral_inhalation)
              info_keys_lower = {str(k).lower(): k for k in drug_info.keys()}

              for flag in bool_flags:
                  matched_key = info_keys_lower.get(flag.lower())
                  if matched_key and matched_key in drug_info:
                      val = drug_info[matched_key]
                      if isinstance(val, bool):
                          drug_flag_records[drug_name][flag].append(val)

      except Exception as e:
          print(f"Warning: Failed to process {file_path.name}: {e}")

  # ==========================================
  # 3. Calculate % True and Format Output Table
  # ==========================================
  summary_rows = []

  for drug_name, flags in drug_flag_records.items():
      row = {"Drug Name": drug_name}
      for flag in bool_flags:
          vals = flags[flag]
          if vals:
              pct_true = round((sum(vals) / len(vals)) * 100, 1)
          else:
              pct_true = 0.0
          row[f"{flag} (% True)"] = pct_true
      summary_rows.append(row)

  df_bool = pd.DataFrame(summary_rows).set_index("Drug Name").sort_index()

  print("\n" + "=" * 90)
  print(f"ADMET BOOLEAN FLAGS SUMMARY (% True Across JSON Runs)")
  print("=" * 90)
  print(df_bool.to_string())

  # ==========================================
  # 4. Save Summary to Tab-Separated CSV
  # ==========================================
  sffx = sffx if 'sffx' in locals() else ""
  output_file = Path(DATA_DIR) / f"admet_boolean_summary{sffx}.csv"
  df_bool.reset_index().to_csv(output_file, sep="\t", index=False)

  print(f"\nSuccessfully saved boolean summary to: {output_file.name}")

  # ==========================================
  # 3. Visualizing Spread Across 5 Metrics
  # ==========================================
  # Create a 2x3 grid to comfortably fit all 5 plots
  plt.style.use("seaborn-v0_8-whitegrid")
  fig, axes = plt.subplots(2, 3, figsize=(18, 10))

  # Flatten axes array for easy indexing
  axes_flat = axes.flatten()

  plot_config = [
      ("Overall Score", "Blues", "Overall Score (0-10)"),
      ("Safety Score", "Reds", "Safety Score (0-10)"),
      ("PK Score", "Greens", "PK Score (0-10)"),
      ("PD Score", "Oranges", "PD Score (0-10)"),
      ("Red Flag Count", "Purples", "Red Flag Count"),
  ]

  for idx, (col_name, palette, x_label) in enumerate(plot_config):
    ax = axes_flat[idx]
    sns.boxplot(data=df_all, x=col_name, y="Drug Name", ax=ax, palette=palette)
    ax.set_title(f"{col_name} Variance", fontweight="bold", fontsize=12)
    ax.set_xlabel(x_label)

    # Only keep y-axis label on leftmost column for cleanliness
    if idx % 3 != 0:
      ax.set_ylabel("")
    else:
      ax.set_ylabel("Drug Name")

  # Hide the unused 6th subplot frame
  fig.delaxes(axes_flat[5])

  # Use fig.suptitle for overall figure title
  fig.suptitle(
      f"Scoring from run #{min(df_all['run_id'])} to #{max(df_all['run_id'])}",
      fontsize=16,
      fontweight="bold",
      y=1.02  # Places the title slightly above the subplots
  )
  plt.tight_layout()
  plt.savefig(DATA_DIR / f"admet_noise_distribution{sffx}.png", dpi=300)
  plt.show()

  return summary, df_all



In [ ]:
DATA_DIR = Path("../PrelResults/ADMET_eval_tests")
summary, df_all = plt_and_summary(DATA_DIR)#, list(range(10, 50)))

In [ ]:
summary

In [ ]:
df_all

In [ ]:
missing = df_all[df_all['Major Red Flags'].str.contains('Missing ', na=False)]
missing['Drug Name'].value_counts()

In [ ]:
missing['run_id'].value_counts()

In [ ]:
from collections import Counter
import json
from pathlib import Path
from typing import Any, Dict, List
import numpy as np
import pandas as pd


def load_run_files(data_dir: Path) -> List[Dict[str, Any]]:
    """Loads all JSON files from DATA_DIR recursively, excluding summary files."""
    records = []
    json_files = [
        f for f in Path(data_dir).glob("**/*.json") if "summary" not in f.name.lower()
    ]

    for file_path in json_files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    records.extend(data)
                elif isinstance(data, dict):
                    records.append(data)
        except Exception as e:
            print(f"Warning: Failed to load {file_path}: {e}")

    return records


def generate_rationales_summary(
    summary_results: Dict[str, Any], csv_output_path: Path
) -> None:
    """Computes the count of unique rationale variations per drug, prints the table,

    and exports it to summary_rationales.csv.
    """
    text_fields = [
        ("overall_prioritization_rationale", "Overall Prioritization"),
        ("safety_rationale", "Safety"),
        ("pharmacokinetics_rationale", "Pharmacokinetics"),
        ("pharmacodynamics_rationale", "Pharmacodynamics"),
    ]

    rows = []
    for drug_name, data in summary_results.items():
        row = {"Drug Name": drug_name}
        unique_rationales = data.get("unique_rationales", {})

        total_vars = 0
        for field_key, col_label in text_fields:
            var_count = len(unique_rationales.get(field_key, []))
            row[f"{col_label} Variations"] = var_count
            total_vars += var_count

        row["Total Unique Variations"] = total_vars
        rows.append(row)

    df_rationales = pd.DataFrame(rows)

    # Print table to console
    print("\n" + "=" * 90)
    print("UNIQUE RATIONALE VARIATIONS PER DRUG")
    print("=" * 90)
    print(df_rationales.to_string(index=False))

    # Export to CSV (tab-separated)
    csv_output_path.parent.mkdir(parents=True, exist_ok=True)
    df_rationales.to_csv(csv_output_path, sep="\t", index=False)
    print(f"\nRationales variation summary saved to: {csv_output_path}")


def write_text_summary(summary_results: Dict[str, Any], output_path: Path) -> None:
    """Writes a human-readable .txt summary containing list_fields and text_fields per drug.

    Header displays Model Avg Score and Overall Score (Mean ± STD).
    """
    lines = []

    list_field_names = [
        "major_red_flags",
        "key_advantages",
        "missing_entries",
        "admin_routes",
        "targeted_viruses",
    ]

    text_field_names = [
        "overall_prioritization_rationale",
        "safety_rationale",
        "pharmacokinetics_rationale",
        "pharmacodynamics_rationale",
    ]

    for drug_name, data in summary_results.items():
        n_runs = data.get("total_runs_found", 0)
        scores = data.get("scores_summary", {})

        # Format Model Avg Score
        model_avg_data = scores.get("model_avg_score", {})
        model_avg_val = model_avg_data.get("mean")
        model_avg_str = (
            f"{model_avg_val:.3f}" if model_avg_val is not None else "N/A"
        )

        # Format Overall Prioritization Score Mean ± STD
        overall_data = scores.get("overall_prioritization_score", {})
        overall_mean = overall_data.get("mean")
        overall_std = overall_data.get("std")

        if overall_mean is not None and overall_std is not None:
            overall_score_str = f"{overall_mean:.1f} ± {overall_std:.1f}"
        elif overall_mean is not None:
            overall_score_str = f"{overall_mean:.1f}"
        else:
            overall_score_str = "N/A"

        lines.append("=" * 90)
        lines.append(
            f"DRUG: {drug_name} (Model Avg Score: {model_avg_str} | Overall Score: {overall_score_str})"
        )
        lines.append("=" * 90)

        # 1. LIST FIELDS SECTION
        lines.append("\n" + "-" * 40)
        lines.append("LIST FIELDS (Frequency & Consensus)")
        lines.append("-" * 40)

        list_counts = data.get("list_counts", {})
        for field in list_field_names:
            clean_field_name = field.replace("_", " ").title()
            lines.append(f"\n[{clean_field_name}]")
            items = list_counts.get(field, [])
            if items:
                for item_info in items:
                    item = item_info["item"]
                    freq = item_info["frequency"]
                    pct = item_info["pct_runs"]
                    lines.append(f"  • {item} ({freq}/{n_runs} runs, {pct}%)")
            else:
                lines.append("  • (None detected)")

        # 2. TEXT FIELDS SECTION WITH MULTI-LINE FORMATTING
        lines.append("\n" + "-" * 40)
        lines.append("TEXT FIELDS (Unique Rationales)")
        lines.append("-" * 40)

        rationales = data.get("unique_rationales", {})
        for field in text_field_names:
            clean_field_name = field.replace("_", " ").title()
            items = rationales.get(field, [])
            lines.append(
                f"\n[{clean_field_name}] ({len(items)} unique variation(s)):"
            )
            if items:
                for idx, item_info in enumerate(items, 1):
                    text = item_info["text"]
                    score_val = item_info.get("score")
                    score_str = (
                        f"Score = {score_val:.1f}; "
                        if score_val is not None
                        else ""
                    )
                    freq = item_info["count"]
                    pct = item_info["pct_runs"]

                    # Formatted header + text body
                    lines.append(
                        f"  {idx}) {score_str}{freq}/{n_runs} runs, {pct}%:"
                    )
                    lines.append(f"{text}\n")
            else:
                lines.append("  • (None provided)")

        lines.append("\n\n")

    # Save to .txt file
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Text summary successfully generated at: {output_path}")


def summarize_drug_data(
    records: List[Dict[str, Any]],
    json_output_path: Path,
    txt_output_path: Path,
    rationales_csv_output_path: Path,
) -> None:
    """Aggregates scores, lists, and rationales per drug into JSON, TXT, and CSV summary files."""

    # Group records by drug name (handles both top-level key dicts and key-value dicts)
    drug_groups: Dict[str, List[Dict[str, Any]]] = {}
    for rec in records:
        if not isinstance(rec, dict):
            continue

        # Check if record is keyed by drug name at top level: {"Alefacept": {...}, "Carmustine": {...}}
        if "drug_name" not in rec and "Drug Name" not in rec:
            for drug_name, drug_info in rec.items():
                if isinstance(drug_info, dict):
                    drug_groups.setdefault(drug_name, []).append(drug_info)
        else:
            # Fallback for flat format: {"drug_name": "Alefacept", ...}
            drug_name = rec.get("drug_name") or rec.get("Drug Name")
            if drug_name:
                drug_groups.setdefault(drug_name, []).append(rec)

    summary_results = {}

    numeric_fields = [
        "overall_prioritization_score",
        "safety_score",
        "pharmacokinetics_score",
        "pharmacodynamics_score",
        "model_avg_score",
    ]

    list_fields = [
        "major_red_flags",
        "key_advantages",
        "missing_entries",
        "admin_routes",
        "targeted_viruses",
    ]

    bool_fields = [
        "chemotherapy",
        "cytotoxic",
        "vaccine",
        "antibody",
        "immunosuppressor",
        "antiinflammatory",
        "IV",
        "oral_inhalation",
        "antiviral",
        "DNA_virus",
        "RNA_virus",
        "endo",
        "respiratory",
        "neuronal",
    ]

    # Map text fields to their associated score fields
    rationale_to_score_field = {
        "overall_prioritization_rationale": "overall_prioritization_score",
        "safety_rationale": "safety_score",
        "pharmacokinetics_rationale": "pharmacokinetics_score",
        "pharmacodynamics_rationale": "pharmacodynamics_score",
    }

    for drug_name, runs in drug_groups.items():
        n_runs = len(runs)

        drug_summary = {
            "total_runs_found": n_runs,
            "scores_summary": {},
            "boolean_flags_consensus": {},
            "list_counts": {},
            "unique_rationales": {},
        }

        # 1. Aggregate Numeric Scores (Mean, Std, Min, Max)
        for field in numeric_fields:
            vals = [
                r[field]
                for r in runs
                if field in r
                and r[field] is not None
                and isinstance(r[field], (int, float))
            ]
            if vals:
                drug_summary["scores_summary"][field] = {
                    "mean": round(float(np.mean(vals)), 3),
                    "std": round(float(np.std(vals)), 3),
                    "min": round(float(np.min(vals)), 3),
                    "max": round(float(np.max(vals)), 3),
                }

        # 2. Boolean Flags (Majority Vote / % True)
        for field in bool_fields:
            vals = [
                r[field]
                for r in runs
                if field in r and isinstance(r[field], bool)
            ]
            if vals:
                pct_true = round((sum(vals) / len(vals)) * 100, 1)
                drug_summary["boolean_flags_consensus"][field] = {
                    "consensus_value": sum(vals) > (len(vals) / 2),
                    "pct_true": pct_true,
                }

        # 3. Deduplicate and Count Frequency for List Items (Red Flags, Advantages, etc.)
        for field in list_fields:
            all_items = []
            for r in runs:
                items = r.get(field, [])
                if isinstance(items, list):
                    all_items.extend(
                        [str(item).strip() for item in items if item]
                    )

            item_counts = Counter(all_items)
            drug_summary["list_counts"][field] = [
                {
                    "item": item,
                    "frequency": count,
                    "pct_runs": round((count / n_runs) * 100, 1),
                }
                for item, count in item_counts.most_common()
            ]

        # 4. Deduplicate Unique Text Rationales & Map Associated Scores + Counts
        for text_field, score_field in rationale_to_score_field.items():
            text_groups = {}  # text -> {"count": int, "scores": list}

            for r in runs:
                text_val = r.get(text_field)
                if isinstance(text_val, str) and text_val.strip():
                    clean_text = text_val.strip()
                    if clean_text not in text_groups:
                        text_groups[clean_text] = {"count": 0, "scores": []}

                    text_groups[clean_text]["count"] += 1

                    # Collect associated score for this rationale if present
                    score_val = r.get(score_field)
                    if score_val is not None and isinstance(
                        score_val, (int, float)
                    ):
                        text_groups[clean_text]["scores"].append(score_val)

            # Build rationale details list
            rationale_list = []
            for text, info in text_groups.items():
                count = info["count"]
                pct = round((count / n_runs) * 100, 1)
                mean_score = (
                    round(float(np.mean(info["scores"])), 1)
                    if info["scores"]
                    else None
                )
                rationale_list.append(
                    {
                        "text": text,
                        "count": count,
                        "pct_runs": pct,
                        "score": mean_score,
                    }
                )

            # Sort by frequency descending (most common variation first)
            rationale_list.sort(key=lambda x: x["count"], reverse=True)
            drug_summary["unique_rationales"][text_field] = rationale_list

        summary_results[drug_name] = drug_summary

    # --- Sort drugs by mean overall_prioritization_score descending ---
    def get_overall_score_mean(item: tuple) -> float:
        _, data = item
        mean_val = (
            data.get("scores_summary", {})
            .get("overall_prioritization_score", {})
            .get("mean")
        )
        return mean_val if mean_val is not None else -float("inf")

    summary_results = dict(
        sorted(summary_results.items(), key=get_overall_score_mean, reverse=True)
    )

    # --- Save JSON summary file ---
    json_output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(summary_results, f, indent=4)

    print(
        f"JSON summary successfully generated for {len(summary_results)} drugs at: {json_output_path}"
    )

    # --- Save TXT summary file (text_fields & list_fields only) ---
    write_text_summary(summary_results, txt_output_path)

    # --- Generate, print, and save Rationales Variation Summary CSV ---
    generate_rationales_summary(summary_results, rationales_csv_output_path)


# ---------------------------------------------------------------------------
# Execution
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    DATA_DIR = Path("../PrelResults/ADMET_eval_tests")
    JSON_OUTPUT = DATA_DIR / "summary.json"
    TXT_OUTPUT = DATA_DIR / "summary.txt"
    RATIONALES_CSV_OUTPUT = DATA_DIR / "summary_rationales.csv"

    records = load_run_files(DATA_DIR)
    if records:
        summarize_drug_data(
            records, JSON_OUTPUT, TXT_OUTPUT, RATIONALES_CSV_OUTPUT
        )
    else:
        print(f"No valid JSON records found in {DATA_DIR}")